# Inhibitory Two-Edge Triad Sensitivity

This notebook consolidates the previous self-free and self-inclusive two-edge triad notebooks into one sensitivity analysis. The self-inclusive variant is shown first, and self-connections are treated as an explicit enrichment and ablation group alongside the Rees motif classes.

## tl;dr

Use `with_self_connections` as the primary local motif readout when diagonal terms should be included. The `self_free` variant is retained as a contrast, but enrichment and ablation sections now include a dedicated `self_connections` group so diagonal terms are tested directly rather than only being appended inside larger motif classes.

## Context & Methods

This analysis should run after the two global notebooks. It does not require their saved outputs, but it uses the same matrix orientation, E/I labeling assumptions, and helper functions. The notebook enumerates induced three-node subgraphs with exactly two non-self directed edges, maps those motifs onto Rees superpatterns, then reruns selected summaries after appending each triad node's nonzero self-connection.

### Key Assumptions

- The source matrix is loaded as rows=presynaptic sources and columns=postsynaptic receivers, then normalized by the helper to rows=receivers and columns=sources.
- Motif labels are assigned from the two non-self edges only. Self-connections are appended after triad selection and are additionally tested as their own enrichment/ablation group.
- The randomized enrichment null for `self_connections` samples nonzero edge positions over the full matrix, including diagonal positions, so diagonal placement is not excluded by construction.
- Null counts use `N_NULL` draws for speed; increase it for final statistical claims.

## Setup

Import helper functions, define matrix paths, and choose the two self-connection variants to compare.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for candidate in (PROJECT_ROOT, PROJECT_ROOT / "notebooks"):
    if str(candidate) not in sys.path:
        sys.path.append(str(candidate))

from inhibitory_modulation import (
    ablate_edges,
    REES_SUPERPATTERN_NAMES,
    TRIAD_CENSUS_TO_REES_SUPERPATTERN,
    enumerate_two_edge_triads,
    load_connectivity,
    randomize_directed_edges,
    stability_summary,
    summarize_blocks,
    summarize_triad_categories,
    triad_ablation_analysis,
    triad_ablation_significance,
    triad_edge_participation,
    triad_enrichment_significance,
    triad_schur_decomposition,
)

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 60)
pd.set_option("display.precision", 4)

CONNECTIVITY_PATH = PROJECT_ROOT / "matrices" / "mij_matrix.csv"
METADATA_NETLIST_PATH = PROJECT_ROOT / "matrices" / "mij_netlist.csv"
MATRIX_ORIENTATION = "pre_by_post"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "03_inhibitory_two_edge_triad_sensitivity"

CONNECTED_ONLY = True
N_NULL = 20
RANDOM_STATE = 0
SCHUR_REGULARIZATION = 1e-6
REGULARIZATION_GRID = np.array([1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1])

VARIANT_CONFIGS = {
    "with_self_connections": True,
    "self_free": False,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONNECTIVITY_PATH


## Data

Load the signed connectivity matrix and E/I labels. The triad enumeration below uses the same loaded matrix for both variants.


In [ ]:
data = load_connectivity(
    CONNECTIVITY_PATH,
    metadata_netlist_path=METADATA_NETLIST_PATH,
    matrix_orientation=MATRIX_ORIENTATION,
)

print(f"Loaded matrix: {data.source}")
print(f"Matrix shape: {data.matrix.shape[0]} x {data.matrix.shape[1]}")
display(data.classes.value_counts().rename(index={"e": "excitatory", "i": "inhibitory"}).to_frame("population_count"))
display(data.matrix.iloc[:5, :5])


## Results

### 1. Enumerate The Two Variants

The self-free variant is the primary motif census. The self-inclusive variant appends diagonal terms to those same selected triads.


In [ ]:
triads_by_variant = {}
overview_rows = []

for variant, include_self_connections in VARIANT_CONFIGS.items():
    triads = enumerate_two_edge_triads(
        data.matrix,
        data.classes,
        connected_only=CONNECTED_ONLY,
        include_self_connections=include_self_connections,
    )
    triads_by_variant[variant] = triads
    edge_count_distribution = triads["edge_count"].value_counts().sort_index().to_dict() if not triads.empty else {}
    overview_rows.append({
        "variant": variant,
        "include_self_connections": include_self_connections,
        "triad_count": len(triads),
        "unique_nonself_edges": int(triad_edge_participation(data.matrix, triads).query("edge_type == 'between_nodes'").shape[0]) if not triads.empty else 0,
        "unique_self_edges": int(triad_edge_participation(data.matrix, triads).query("edge_type == 'self'").shape[0]) if not triads.empty else 0,
        "edge_count_distribution": edge_count_distribution,
    })

overview = pd.DataFrame(overview_rows)
display(overview)


### 2. Compare Motif And E/I Composition

Motif labels should match across variants because they are based on non-self edges. Weight and local spectral summaries can differ once self-connections are appended.


In [ ]:
motif_summary = []
node_ei_summary = []
edge_ei_summary = []

for variant, triads in triads_by_variant.items():
    motif_table = summarize_triad_categories(
        triads,
        ["rees_superpattern", "motif", "rees_superpattern_name"],
    ).reset_index()
    motif_table.insert(0, "variant", variant)
    motif_summary.append(motif_table)

    node_table = summarize_triad_categories(triads, "node_ei_signature").reset_index()
    node_table.insert(0, "variant", variant)
    node_ei_summary.append(node_table)

    edge_table = summarize_triad_categories(triads, "edge_ei_signature").reset_index()
    edge_table.insert(0, "variant", variant)
    edge_ei_summary.append(edge_table)

motif_summary = pd.concat(motif_summary, ignore_index=True)
node_ei_summary = pd.concat(node_ei_summary, ignore_index=True)
edge_ei_summary = pd.concat(edge_ei_summary, ignore_index=True)

print("Rees motif summary")
display(motif_summary)
print("Node E/I composition")
display(node_ei_summary)
print("Top directed edge E/I signatures")
display(edge_ei_summary.groupby("variant", group_keys=False).head(12))


### 3. Region Signatures

Region signatures collapse each triad to the unique anatomical labels represented by its three populations.


In [ ]:
region_summary = []
within_region = []

for variant, triads in triads_by_variant.items():
    table = summarize_triad_categories(triads, "region_signature").reset_index()
    table.insert(0, "variant", variant)
    region_summary.append(table)

    rates = (
        triads.groupby("motif")
        .agg(
            triad_count=("triad_id", "count"),
            within_region_rate=("within_region", "mean"),
            median_region_count=("region_count", "median"),
        )
        .reset_index()
    )
    rates.insert(0, "variant", variant)
    within_region.append(rates)

region_summary = pd.concat(region_summary, ignore_index=True)
within_region = pd.concat(within_region, ignore_index=True)

print("Most frequent region signatures")
display(region_summary.groupby("variant", group_keys=False).head(15))
print("Within-region share by motif")
display(within_region.sort_values(["variant", "triad_count"], ascending=[True, False]))


### 4. Rank Triads And Participating Edges

This is where self-connections matter most: the same base motif can gain diagonal edges that affect total weight and edge participation.


In [ ]:
ranked_triads = []
edge_participation = []
ranking_columns = [
    "variant",
    "triad_id",
    "motif",
    "rees_superpattern",
    "rees_superpattern_name",
    "node_ei_signature",
    "edge_ei_signature",
    "region_signature",
    "node_a",
    "node_b",
    "node_c",
    "edge_count",
    "self_edge_count",
    "total_abs_weight",
]

for variant, triads in triads_by_variant.items():
    ranked = triads.sort_values("total_abs_weight", ascending=False).copy()
    ranked.insert(0, "variant", variant)
    ranked_triads.append(ranked[ranking_columns])

    participation = triad_edge_participation(data.matrix, triads)
    participation.insert(0, "variant", variant)
    edge_participation.append(participation)

ranked_triads = pd.concat(ranked_triads, ignore_index=True)
edge_participation = pd.concat(edge_participation, ignore_index=True)

print("Top triads by absolute weight")
display(ranked_triads.groupby("variant", group_keys=False).head(15))
print("Top participating directed edges")
display(edge_participation.groupby("variant", group_keys=False).head(15))


### 5. Enrichment Significance

Run a compact randomized-edge null for Rees motif labels, node E/I composition, and the explicit `self_connections` group. Increase `N_NULL` before relying on p-values in a manuscript or presentation.

In [ ]:
def self_connection_enrichment_significance(matrix, n_null=N_NULL, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    arr = matrix.to_numpy(dtype=float)
    observed_count = int(np.count_nonzero(np.diag(arr)))
    observed_nonzero_count = int(np.count_nonzero(arr))
    flat_size = arr.size
    null_counts = []
    for _ in range(n_null):
        sampled = np.zeros_like(arr)
        sampled_positions = rng.choice(flat_size, size=observed_nonzero_count, replace=False)
        sampled.flat[sampled_positions] = 1.0
        null_counts.append(int(np.count_nonzero(np.diag(sampled))))
    null_values = np.array(null_counts, dtype=float)
    null_sd = float(null_values.std(ddof=1)) if len(null_values) > 1 else 0.0
    return pd.DataFrame(
        [
            {
                "rees_superpattern": "self_connections",
                "observed_count": observed_count,
                "null_mean": float(null_values.mean()) if len(null_values) else np.nan,
                "null_sd": null_sd,
                "null_p95": float(np.quantile(null_values, 0.95)) if len(null_values) else np.nan,
                "enrichment_ratio": observed_count / null_values.mean() if len(null_values) and null_values.mean() else np.inf,
                "z_score": (observed_count - null_values.mean()) / null_sd if null_sd else np.nan,
                "empirical_p_ge": float((np.sum(null_values >= observed_count) + 1) / (len(null_values) + 1)) if len(null_values) else np.nan,
                "significant_enriched": bool(len(null_values) and observed_count > np.quantile(null_values, 0.95)),
                "rees_superpattern_name": "diagonal self-connections",
            }
        ]
    )


enrichment_motif = []
enrichment_node_ei = []

for offset, (variant, include_self_connections) in enumerate(VARIANT_CONFIGS.items()):
    motif_table = triad_enrichment_significance(
        data.matrix,
        data.classes,
        by="rees_superpattern",
        n_null=N_NULL,
        random_state=RANDOM_STATE + offset,
        connected_only=CONNECTED_ONLY,
        include_self_connections=include_self_connections,
    )
    motif_table.insert(0, "variant", variant)
    motif_table["rees_superpattern_name"] = motif_table["rees_superpattern"].map(REES_SUPERPATTERN_NAMES)
    motif_table["rees_superpattern_name"] = motif_table["rees_superpattern_name"].fillna(motif_table["rees_superpattern"])
    self_table = self_connection_enrichment_significance(
        data.matrix,
        n_null=N_NULL,
        random_state=RANDOM_STATE + 100 + offset,
    )
    self_table.insert(0, "variant", variant)
    enrichment_motif.append(pd.concat([motif_table, self_table], ignore_index=True, sort=False))

    node_table = triad_enrichment_significance(
        data.matrix,
        data.classes,
        by="node_ei_signature",
        n_null=N_NULL,
        random_state=RANDOM_STATE + 10 + offset,
        connected_only=CONNECTED_ONLY,
        include_self_connections=include_self_connections,
    )
    node_table.insert(0, "variant", variant)
    enrichment_node_ei.append(node_table)

enrichment_motif = pd.concat(enrichment_motif, ignore_index=True)
enrichment_node_ei = pd.concat(enrichment_node_ei, ignore_index=True)

print(f"Null draws per enrichment table: {N_NULL}")
display(enrichment_motif)
display(enrichment_node_ei)

### 6. Ablation Significance

Ablation removes all triad-participating edges, each Rees-superpattern edge set, and the explicit self-connection group. The self-connection null samples random nonzero edge sets of the same size from the full matrix, including diagonal candidates.

In [ ]:
def self_connection_ablation_rows(matrix, n_null=N_NULL, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    base = stability_summary(matrix, "baseline")
    self_edges = [
        (node, node)
        for node in matrix.index
        if float(matrix.loc[node, node]) != 0
    ]
    observed_summary = stability_summary(ablate_edges(matrix, self_edges), "observed_self_connections")
    observed_delta = float(observed_summary["spectral_radius"] - base["spectral_radius"])
    observed_edges = [
        (pre, post)
        for pre in matrix.columns
        for post in matrix.index
        if float(matrix.loc[post, pre]) != 0
    ]
    null_deltas = []
    for _ in range(n_null):
        sampled_indices = rng.choice(len(observed_edges), size=len(self_edges), replace=False)
        sampled_edges = [observed_edges[index] for index in sampled_indices]
        null_summary = stability_summary(ablate_edges(matrix, sampled_edges), "null")
        null_deltas.append(float(null_summary["spectral_radius"] - base["spectral_radius"]))
    null_values = np.array(null_deltas, dtype=float)
    null_row = {
        "category": "self_connections",
        "ablated_unique_edges": len(self_edges),
        "observed_delta_spectral_radius": observed_delta,
        "null_mean_delta": float(null_values.mean()) if len(null_values) else np.nan,
        "null_p05_delta": float(np.quantile(null_values, 0.05)) if len(null_values) else np.nan,
        "null_p95_delta": float(np.quantile(null_values, 0.95)) if len(null_values) else np.nan,
        "empirical_p_more_stabilizing": float((np.sum(null_values <= observed_delta) + 1) / (len(null_values) + 1)) if len(null_values) else np.nan,
        "empirical_p_more_destabilizing": float((np.sum(null_values >= observed_delta) + 1) / (len(null_values) + 1)) if len(null_values) else np.nan,
        "rees_superpattern_name": "diagonal self-connections",
    }
    ablation_row = pd.Series(
        {
            "name": "ablate_self_connections",
            **observed_summary,
            "ablated_unique_edges": len(self_edges),
            "delta_max_real": observed_summary["max_real"] - base["max_real"],
            "delta_spectral_radius": observed_summary["spectral_radius"] - base["spectral_radius"],
        }
    )
    return ablation_row, pd.DataFrame([null_row])


global_ablation = []
motif_ablation = []
motif_ablation_significance = []

for offset, (variant, triads) in enumerate(triads_by_variant.items()):
    global_table = triad_ablation_analysis(data.matrix, triads).reset_index()
    global_table.insert(0, "variant", variant)
    global_ablation.append(global_table)

    top_superpatterns = (
        motif_summary.loc[motif_summary["variant"] == variant, "rees_superpattern"]
        .drop_duplicates()
        .tolist()
    )
    motif_table = triad_ablation_analysis(
        data.matrix,
        triads,
        category_col="rees_superpattern",
        categories=top_superpatterns,
    ).reset_index()
    self_ablation_row, self_sig_table = self_connection_ablation_rows(
        data.matrix,
        n_null=N_NULL,
        random_state=RANDOM_STATE + 200 + offset,
    )
    motif_table = pd.concat([motif_table, self_ablation_row.to_frame().T], ignore_index=True, sort=False)
    motif_table.insert(0, "variant", variant)
    motif_ablation.append(motif_table)

    sig_table = triad_ablation_significance(
        data.matrix,
        triads,
        category_col="rees_superpattern",
        categories=top_superpatterns,
        n_null=N_NULL,
        random_state=RANDOM_STATE + 20 + offset,
    )
    sig_table.insert(0, "variant", variant)
    sig_table["rees_superpattern_name"] = sig_table["category"].map(REES_SUPERPATTERN_NAMES)
    sig_table["rees_superpattern_name"] = sig_table["rees_superpattern_name"].fillna(sig_table["category"])
    self_sig_table.insert(0, "variant", variant)
    motif_ablation_significance.append(pd.concat([sig_table, self_sig_table], ignore_index=True, sort=False))

global_ablation = pd.concat(global_ablation, ignore_index=True)
motif_ablation = pd.concat(motif_ablation, ignore_index=True)
motif_ablation_significance = pd.concat(motif_ablation_significance, ignore_index=True)

print("All triad-participating edge ablations")
display(global_ablation)
print("Rees-superpattern and self-connection edge ablations")
display(motif_ablation)
print(f"Random edge-set null draws per superpattern/self group: {N_NULL}")
display(motif_ablation_significance)

### 7. Schur Reduction Sensitivity

Apply the E/I Schur-complement reduction to the triad aggregate and sweep the inhibitory-block regularization. Use this as a diagnostic ranking, especially when self-connections are included.


In [ ]:
schur_summary = []
schur_sensitivity = []

for variant, triads in triads_by_variant.items():
    triad_schur = triad_schur_decomposition(
        data.matrix,
        data.classes,
        triads,
        regularization=SCHUR_REGULARIZATION,
        normalize="triad_count",
    )
    block_table = summarize_blocks(triad_schur.blocks).reset_index()
    block_table.insert(0, "variant", variant)
    schur_summary.append(block_table)

    for regularization in REGULARIZATION_GRID:
        decomp = triad_schur_decomposition(
            data.matrix,
            data.classes,
            triads,
            regularization=float(regularization),
            normalize="triad_count",
        )
        row = decomp.effective_stability.copy()
        row["variant"] = variant
        row["regularization"] = regularization
        schur_sensitivity.append(row)

schur_summary = pd.concat(schur_summary, ignore_index=True)
schur_sensitivity = pd.DataFrame(schur_sensitivity)

print("Triad-aggregate E/I block summaries")
display(schur_summary)
print("Schur regularization sensitivity")
display(schur_sensitivity.set_index(["variant", "regularization"]))


### 8. Save Tables

Save the consolidated sensitivity outputs so the triad analysis has the same handoff pattern as the two global notebooks.


In [ ]:
overview.to_csv(OUTPUT_DIR / "triad_variant_overview.csv", index=False)
motif_summary.to_csv(OUTPUT_DIR / "triad_motif_summary_by_variant.csv", index=False)
node_ei_summary.to_csv(OUTPUT_DIR / "triad_node_ei_summary_by_variant.csv", index=False)
edge_ei_summary.to_csv(OUTPUT_DIR / "triad_edge_ei_summary_by_variant.csv", index=False)
region_summary.to_csv(OUTPUT_DIR / "triad_region_summary_by_variant.csv", index=False)
within_region.to_csv(OUTPUT_DIR / "triad_within_region_by_motif.csv", index=False)
ranked_triads.to_csv(OUTPUT_DIR / "top_triads_by_variant.csv", index=False)
edge_participation.to_csv(OUTPUT_DIR / "triad_edge_participation_by_variant.csv", index=False)
enrichment_motif.to_csv(OUTPUT_DIR / "triad_motif_enrichment_by_variant.csv", index=False)
enrichment_node_ei.to_csv(OUTPUT_DIR / "triad_node_ei_enrichment_by_variant.csv", index=False)
global_ablation.to_csv(OUTPUT_DIR / "triad_global_ablation_by_variant.csv", index=False)
motif_ablation.to_csv(OUTPUT_DIR / "triad_motif_ablation_by_variant.csv", index=False)
motif_ablation_significance.to_csv(OUTPUT_DIR / "triad_motif_ablation_significance_by_variant.csv", index=False)
schur_summary.to_csv(OUTPUT_DIR / "triad_schur_block_summary_by_variant.csv", index=False)
schur_sensitivity.to_csv(OUTPUT_DIR / "triad_schur_regularization_sensitivity_by_variant.csv", index=False)

for variant, triads in triads_by_variant.items():
    triads.to_csv(OUTPUT_DIR / f"enumerated_triads_{variant}.csv", index=False)

print(f"Saved consolidated triad sensitivity tables to {OUTPUT_DIR.resolve()}")


## Takeaways

- The self-inclusive variant is now the first/local primary view, with the self-free variant retained as a sensitivity contrast.
- Self-connections are included in ranking, enrichment, and ablation outputs as their own tested group.
- Rees motif labels remain based on two non-self edges, so the self-connection group should be interpreted as a diagonal-edge effect rather than a separate Rees triad topology.

## References

- Holland, P. W., & Leinhardt, S. (1974). *The Statistical Analysis of Local Structure in Social Networks*. NBER Working Paper 0044. https://doi.org/10.3386/w0044
- NetworkX documentation: `triadic_census`, including the 16 directed triad-census labels used here. https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.triads.triadic_census.html
- Rees, C. L., Wheeler, D. W., Hamilton, D. J., White, C. M., Komendantov, A. O., & Ascoli, G. A. (2016). *Graph Theoretic and Motif Analyses of the Hippocampal Neuron Type Potential Connectome*. eNeuro, 3(6), ENEURO.0205-16.2016. https://www.eneuro.org/content/3/6/ENEURO.0205-16.2016
- Zhang, F. (Ed.). (2005). *The Schur Complement and Its Applications*. Springer. https://doi.org/10.1007/b105056
- Golub, G. H., & Van Loan, C. F. (2013). *Matrix Computations* (4th ed.). Johns Hopkins University Press. https://www.press.jhu.edu/books/title/10678/matrix-computations
